# 11 — Sensitivity Analysis

This notebook comes after:

`10_Recovery_Validation.ipynb`

Purpose:

Consolidate the project sensitivity checks into one review layer.

This notebook does:

- read variable-selection diagnostics from notebook 06;
- read profile-level POSet outputs from notebook 07;
- read epsilon tolerance outputs from notebook 08;
- read epsilon-margin robustness outputs from notebook 09;
- read recovery-validation outputs from notebook 10;
- summarize sensitivity across:
  - variable set;
  - discretization level;
  - epsilon tolerance;
  - epsilon margin;
  - recovery-validation variants;
- create compact report-ready tables;
- create decision-candidate tables for final methodological choices.

This notebook does **not**:

- make final methodological decisions automatically;
- create a scalar Economic Resilience Index;
- turn POSet layers into a definitive ranking;
- use recovery as an ordering variable.

Important:

The output of this notebook is a **decision-support layer**.  
It tells us where the analysis is stable, where it is sensitive, and which decisions still need to be made before final reporting.

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
PROFILE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
EPSILON_TOLERANCE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Sensitivity_Country_Level"
EPSILON_MARGIN_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Margin_POSet_Robustness"
RECOVERY_VALIDATION_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Sensitivity_Analysis"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "11_Sensitivity_Analysis"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Sensitivity_Analysis"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Pre-POSet folder:", PRE_POSET_DIR.resolve())
print("Profile POSet folder:", PROFILE_POSET_DIR.resolve())
print("Epsilon tolerance folder:", EPSILON_TOLERANCE_DIR.resolve())
print("Epsilon margin folder:", EPSILON_MARGIN_DIR.resolve())
print("Recovery validation folder:", RECOVERY_VALIDATION_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Figures folder:", FIGURES_DIR.resolve())

Run ID: 20260622_083940
Pre-POSet folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA
Profile POSet folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Profile_POSet_Main
Epsilon tolerance folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Epsilon_Sensitivity_Country_Level
Epsilon margin folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Epsilon_Margin_POSet_Robustness
Recovery validation folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery_Validation
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q0

In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

MAIN_SET_NAME = "baseline_6_variables"
WORKING_MAIN_PROFILE_LEVEL = 5

# Sensitivity values to highlight in final compact tables.
PROFILE_LEVELS_TO_REVIEW = [3, 4, 5]
EPSILON_VALUES_TO_REVIEW = [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]
EPSILON_MARGIN_VALUES_TO_REVIEW = [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]

# Recovery validation variants to highlight.
PRIMARY_RECOVERY_VARIANT = "all_recovery_available"
STRICT_RECOVERY_VARIANT = "affected_recovered_only_exclude_0_and_20"

# Report-readiness thresholds.
# These are heuristic flags, not hard rules.
MIN_COUNTRY_COUNT_GOOD = 25
MAX_PARETO_SHARE_REVIEW = 0.45
MIN_COMPARABILITY_REVIEW = 0.10
MAX_COMPARABILITY_REVIEW = 0.70

print("Main set:", MAIN_SET_NAME)
print("Working main profile level:", WORKING_MAIN_PROFILE_LEVEL)
print("Primary recovery variant:", PRIMARY_RECOVERY_VARIANT)
print("Strict recovery variant:", STRICT_RECOVERY_VARIANT)

Main set: baseline_6_variables
Working main profile level: 5
Primary recovery variant: all_recovery_available
Strict recovery variant: affected_recovered_only_exclude_0_and_20


In [3]:
# ------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------

def read_csv_if_exists(path, label, required=False):
    path = Path(path)

    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {label}: {len(df)} rows")
        return df

    if required:
        raise FileNotFoundError(f"{label} not found: {path}")

    print(f"Optional file not found: {label} -> {path}")
    return pd.DataFrame()


def clean_shock(df):
    out = df.copy()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.strip()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    return out


def as_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def safe_ratio(numerator, denominator):
    if pd.isna(denominator) or denominator == 0:
        return np.nan
    return numerator / denominator


def jaccard_from_semicolon_sets(a, b):
    set_a = set([x for x in str(a).split(";") if x and x != "nan"])
    set_b = set([x for x in str(b).split(";") if x and x != "nan"])

    if not set_a and not set_b:
        return np.nan

    return len(set_a & set_b) / len(set_a | set_b)


def numeric_or_nan(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def classify_metric_direction(value, good_when="higher", weak=None, strong=None):
    if pd.isna(value):
        return "missing"

    if good_when == "higher":
        if strong is not None and value >= strong:
            return "strong"
        if weak is not None and value < weak:
            return "weak"
        return "review"
    else:
        if strong is not None and value <= strong:
            return "strong"
        if weak is not None and value > weak:
            return "weak"
        return "review"


def add_metadata(table, metadata):
    out = table.copy()

    for col, val in metadata.items():
        if col in out.columns:
            out[col] = val
        else:
            out.insert(0, col, val)

    order = list(metadata.keys())
    cols = [col for col in order if col in out.columns]
    cols += [col for col in out.columns if col not in cols]

    return out[cols].copy()


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


def make_status_note(row):
    notes = []

    if "country_count" in row and pd.notna(row["country_count"]):
        if row["country_count"] < MIN_COUNTRY_COUNT_GOOD:
            notes.append("small sample")

    if "pareto_share" in row and pd.notna(row["pareto_share"]):
        if row["pareto_share"] > MAX_PARETO_SHARE_REVIEW:
            notes.append("large frontier")

    if "comparability_ratio" in row and pd.notna(row["comparability_ratio"]):
        if row["comparability_ratio"] < MIN_COMPARABILITY_REVIEW:
            notes.append("very low comparability")
        if row["comparability_ratio"] > MAX_COMPARABILITY_REVIEW:
            notes.append("very high comparability")

    if not notes:
        return "no immediate warning"

    return "; ".join(notes)

In [4]:
# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

# Notebook 06
candidate_variable_scorecard = read_csv_if_exists(
    PRE_POSET_DIR / "candidate_variable_scorecard.csv",
    "candidate_variable_scorecard",
)

candidate_variable_redundancy_pairs = read_csv_if_exists(
    PRE_POSET_DIR / "candidate_variable_redundancy_pairs.csv",
    "candidate_variable_redundancy_pairs",
)

candidate_variable_sets = read_csv_if_exists(
    PRE_POSET_DIR / "candidate_variable_sets.csv",
    "candidate_variable_sets",
)

baseline_complete_case_sample = read_csv_if_exists(
    PRE_POSET_DIR / "baseline_complete_case_sample_by_shock.csv",
    "baseline_complete_case_sample_by_shock",
)

# Notebook 07
profile_run_summary = read_csv_if_exists(
    PROFILE_POSET_DIR / "profile_run_summary.csv",
    "profile_run_summary",
    required=True,
)

profile_poset_quality_diagnostics = read_csv_if_exists(
    Path("Data/Diagnostics/07_Profile_POSet_Main/profile_poset_quality_diagnostics.csv"),
    "profile_poset_quality_diagnostics",
)

profile_pareto_countries = read_csv_if_exists(
    PROFILE_POSET_DIR / "profile_pareto_countries.csv",
    "profile_pareto_countries",
)

profile_layer_summary = read_csv_if_exists(
    PROFILE_POSET_DIR / "profile_layer_summary.csv",
    "profile_layer_summary",
)

# Notebook 08
epsilon_run_summary = read_csv_if_exists(
    EPSILON_TOLERANCE_DIR / "epsilon_run_summary.csv",
    "epsilon_run_summary",
)

epsilon_cycle_diagnostics = read_csv_if_exists(
    EPSILON_TOLERANCE_DIR / "epsilon_cycle_diagnostics.csv",
    "epsilon_cycle_diagnostics",
)

epsilon_frontier_stability_summary = read_csv_if_exists(
    EPSILON_TOLERANCE_DIR / "epsilon_frontier_stability_summary.csv",
    "epsilon_frontier_stability_summary",
)

# Notebook 09
epsilon_margin_run_summary = read_csv_if_exists(
    EPSILON_MARGIN_DIR / "epsilon_margin_run_summary.csv",
    "epsilon_margin_run_summary",
)

epsilon_margin_validity_diagnostics = read_csv_if_exists(
    EPSILON_MARGIN_DIR / "epsilon_margin_validity_diagnostics.csv",
    "epsilon_margin_validity_diagnostics",
)

epsilon_margin_frontier_stability_summary = read_csv_if_exists(
    EPSILON_MARGIN_DIR / "epsilon_margin_frontier_stability_summary.csv",
    "epsilon_margin_frontier_stability_summary",
)

epsilon_tolerance_vs_margin_comparison = read_csv_if_exists(
    EPSILON_MARGIN_DIR / "epsilon_tolerance_vs_margin_comparison.csv",
    "epsilon_tolerance_vs_margin_comparison",
)

# Notebook 10
profile_recovery_validation_summary = read_csv_if_exists(
    RECOVERY_VALIDATION_DIR / "profile_recovery_validation_summary.csv",
    "profile_recovery_validation_summary",
)

working_main_profile_recovery_validation_summary = read_csv_if_exists(
    RECOVERY_VALIDATION_DIR / "working_main_profile_recovery_validation_summary.csv",
    "working_main_profile_recovery_validation_summary",
)

epsilon_margin_recovery_validation_summary = read_csv_if_exists(
    RECOVERY_VALIDATION_DIR / "epsilon_margin_recovery_validation_summary.csv",
    "epsilon_margin_recovery_validation_summary",
)

working_main_epsilon_margin_recovery_validation_summary = read_csv_if_exists(
    RECOVERY_VALIDATION_DIR / "working_main_epsilon_margin_recovery_validation_summary.csv",
    "working_main_epsilon_margin_recovery_validation_summary",
)

recovery_validation_takeaway_table = read_csv_if_exists(
    RECOVERY_VALIDATION_DIR / "recovery_validation_takeaway_table.csv",
    "recovery_validation_takeaway_table",
)

recovery_interpretation_cases = read_csv_if_exists(
    RECOVERY_VALIDATION_DIR / "recovery_interpretation_cases.csv",
    "recovery_interpretation_cases",
)

# Clean key columns.
for df_name in [
    "baseline_complete_case_sample",
    "profile_run_summary",
    "profile_poset_quality_diagnostics",
    "profile_pareto_countries",
    "profile_layer_summary",
    "epsilon_run_summary",
    "epsilon_cycle_diagnostics",
    "epsilon_frontier_stability_summary",
    "epsilon_margin_run_summary",
    "epsilon_margin_validity_diagnostics",
    "epsilon_margin_frontier_stability_summary",
    "epsilon_tolerance_vs_margin_comparison",
    "profile_recovery_validation_summary",
    "working_main_profile_recovery_validation_summary",
    "epsilon_margin_recovery_validation_summary",
    "working_main_epsilon_margin_recovery_validation_summary",
    "recovery_validation_takeaway_table",
    "recovery_interpretation_cases",
]:
    df = globals().get(df_name, pd.DataFrame())
    if isinstance(df, pd.DataFrame) and not df.empty:
        globals()[df_name] = clean_shock(df)

input_summary = pd.DataFrame([
    {"input": "candidate_variable_scorecard", "rows": len(candidate_variable_scorecard), "columns": len(candidate_variable_scorecard.columns)},
    {"input": "candidate_variable_redundancy_pairs", "rows": len(candidate_variable_redundancy_pairs), "columns": len(candidate_variable_redundancy_pairs.columns)},
    {"input": "profile_run_summary", "rows": len(profile_run_summary), "columns": len(profile_run_summary.columns)},
    {"input": "epsilon_run_summary", "rows": len(epsilon_run_summary), "columns": len(epsilon_run_summary.columns)},
    {"input": "epsilon_margin_run_summary", "rows": len(epsilon_margin_run_summary), "columns": len(epsilon_margin_run_summary.columns)},
    {"input": "recovery_validation_takeaway_table", "rows": len(recovery_validation_takeaway_table), "columns": len(recovery_validation_takeaway_table.columns)},
])

input_summary.to_csv(
    DIAGNOSTICS_DIR / "sensitivity_input_summary.csv",
    index=False,
)

display(input_summary)

Loaded candidate_variable_scorecard: 16 rows
Optional file not found: candidate_variable_redundancy_pairs -> D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA\candidate_variable_redundancy_pairs.csv
Loaded candidate_variable_sets: 5 rows
Loaded baseline_complete_case_sample_by_shock: 57 rows
Loaded profile_run_summary: 12 rows
Loaded profile_poset_quality_diagnostics: 12 rows
Loaded profile_pareto_countries: 120 rows
Loaded profile_layer_summary: 61 rows
Loaded epsilon_run_summary: 48 rows
Loaded epsilon_cycle_diagnostics: 48 rows
Loaded epsilon_frontier_stability_summary: 48 rows
Loaded epsilon_margin_run_summary: 48 rows
Loaded epsilon_margin_validity_diagnostics: 48 rows
Loaded epsilon_margin_frontier_stability_summary: 48 rows
Loaded epsilon_tolerance_vs_margin_comparison: 48 rows
Loaded profile_recovery_validation_summary: 48 rows
Loaded working_main_profile_recovery_validation_summary: 8 r

,input,rows,columns
0,candidate_variable_scorecard,16,20
1,candidate_variable_redundancy_pairs,0,0
2,profile_run_summary,12,23
3,epsilon_run_summary,48,23
4,epsilon_margin_run_summary,48,23
5,recovery_validation_takeaway_table,16,11


In [5]:
# ------------------------------------------------------
# 1. VARIABLE SELECTION SENSITIVITY
# ------------------------------------------------------

variable_selection_sensitivity = pd.DataFrame()

if not candidate_variable_scorecard.empty:
    variable_selection_sensitivity = candidate_variable_scorecard.copy()

    # Create soft report flags where columns are present.
    if "min_analysis_coverage_rate" in variable_selection_sensitivity.columns:
        variable_selection_sensitivity["coverage_flag"] = np.where(
            variable_selection_sensitivity["min_analysis_coverage_rate"] >= 0.90,
            "strong",
            np.where(
                variable_selection_sensitivity["min_analysis_coverage_rate"] >= 0.70,
                "acceptable_but_review",
                "weak_review",
            ),
        )

    if "max_abs_correlation_with_other_candidates" in variable_selection_sensitivity.columns:
        variable_selection_sensitivity["redundancy_flag"] = np.where(
            variable_selection_sensitivity["max_abs_correlation_with_other_candidates"] >= 0.90,
            "high_redundancy",
            np.where(
                variable_selection_sensitivity["max_abs_correlation_with_other_candidates"] >= 0.75,
                "moderate_redundancy",
                "low_redundancy",
            ),
        )

    if "recommended_role" in variable_selection_sensitivity.columns:
        variable_selection_sensitivity["role_review_note"] = variable_selection_sensitivity["recommended_role"].astype(str)
    else:
        variable_selection_sensitivity["role_review_note"] = ""

if not candidate_variable_redundancy_pairs.empty:
    redundancy_pair_summary = candidate_variable_redundancy_pairs.copy()
else:
    redundancy_pair_summary = pd.DataFrame(columns=[
        "shock_id",
        "variable_a",
        "variable_b",
        "correlation",
        "abs_correlation",
        "redundancy_flag",
    ])

variable_selection_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_variable_selection_scorecard.csv",
    index=False,
)

redundancy_pair_summary.to_csv(
    PROCESSED_DIR / "sensitivity_variable_redundancy_pairs.csv",
    index=False,
)

display(variable_selection_sensitivity.head(100))
display(redundancy_pair_summary.head(100))

,candidate_variable,candidate_variable_raw,source_column,concept,domain,priority,direction,reason,recommended_use,min_coverage_rate_broad_master,mean_coverage_rate_broad_master,min_countries_non_missing_broad_master,min_analysis_coverage_rate,mean_analysis_coverage_rate,min_analysis_countries_non_missing,shocks_available,max_abs_correlation,max_redundancy_class,pre_poset_recommendation,warning,coverage_flag,role_review_note
0,energy_security_score_0_100,energy_security_aligned_raw,energy_import_dependency_raw,Energy security,external_energy_resilience,baseline_candidate,higher_is_better,Lower energy import dependency means higher en...,main_or_baseline,0.8896,0.8900,130,0.9767,0.9770,42,2,0.3162,lower_redundancy_risk,recommended_baseline_pool,NaN,strong,
1,human_capital_tertiary_score_0_100,human_capital_tertiary_aligned_raw,tertiary_education_25_64,Adult tertiary educational attainment,human_capital_capacity,baseline_candidate,higher_is_better,Adult tertiary attainment captures accumulated...,main_or_baseline,0.2534,0.2728,37,0.8409,0.8972,37,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,NaN,acceptable_but_review,
2,rd_intensity_score_0_100,rd_intensity_aligned_raw,rd_gdp,R&D intensity,innovation_capacity,baseline_candidate,higher_is_better,R&D/GDP captures innovation capacity and is al...,main_or_baseline,0.2922,0.2968,44,0.8636,0.8737,38,2,0.6869,lower_redundancy_risk,recommended_baseline_pool,NaN,acceptable_but_review,
3,governance_capacity_score_0_100,governance_capacity_aligned_raw,wgi_mahalanobis_ideal_score_full_wgi,Governance capacity,institutional_capacity,baseline_candidate,higher_is_better,Derived WGI governance composite summarizes in...,main_or_baseline,0.9863,0.9932,144,1.0000,1.0000,43,2,0.5497,lower_redundancy_risk,recommended_baseline_pool,NaN,strong,
4,employment_strength_score_0_100,employment_strength_aligned_raw,unemployment_rate,Employment strength,labour_market_capacity,baseline_candidate,higher_is_better,Lower unemployment indicates stronger labour-m...,main_or_baseline,0.3356,0.3366,49,0.9545,0.9656,42,2,0.4342,lower_redundancy_risk,recommended_baseline_pool,NaN,strong,
5,gdp_growth_stability_pre_2007_score_0_100,gdp_growth_stability_pre_2007_aligned_raw,gdp_growth_stability_negative_std_pre_2007,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.3014,0.3014,44,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,recommended_baseline_pool,NaN,strong,
6,gdp_growth_stability_pre_2019_score_0_100,gdp_growth_stability_pre_2019_aligned_raw,gdp_growth_stability_negative_std_pre_2019,GDP growth stability,macro_stability,baseline_candidate,higher_is_better,Pre-shock GDP growth volatility converted to s...,main_or_baseline,0.2857,0.2857,44,1.0000,1.0000,43,1,0.7370,moderate_redundancy_risk,recommended_baseline_pool,NaN,strong,
7,unemployment_stability_pre_2007_score_0_100,unemployment_stability_pre_2007_aligned_raw,unemployment_stability_negative_std_pre_2007,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3288,0.3288,48,0.9545,0.9545,42,1,0.4128,lower_redundancy_risk,review_candidate,NaN,strong,
8,unemployment_stability_pre_2019_score_0_100,unemployment_stability_pre_2019_aligned_raw,unemployment_stability_negative_std_pre_2019,Unemployment stability,labour_market_capacity,review_candidate,higher_is_better,Pre-shock unemployment volatility converted to...,review_or_sensitivity,0.3312,0.3312,51,0.9535,0.9535,41,1,0.6411,lower_redundancy_risk,review_candidate,NaN,strong,
9,inflation_stability_pre_2007_score_0_100,inflation_stability_pre_2007_aligned_raw,inflation_stability_negative_std_pre_2007,Inflation stability,macro_stability,review_candidate,higher_is_better,Pre-shock inflation volatility converted to st...,review_or_sensitivity,0.3219,0.3219,47,1.0000,1.0000,44,1,0.6874,lower_redundancy_risk,review_candida

,shock_id,variable_a,variable_b,correlation,abs_correlation,redundancy_flag


In [6]:
# ------------------------------------------------------
# 2. PROFILE POSET / DISCRETIZATION SENSITIVITY
# ------------------------------------------------------

profile_discretization_sensitivity = profile_run_summary.copy()

if not profile_discretization_sensitivity.empty:
    profile_discretization_sensitivity["pareto_share"] = (
        profile_discretization_sensitivity["pareto_country_count"]
        / profile_discretization_sensitivity["country_count"]
    )

    profile_discretization_sensitivity["profile_per_country_ratio"] = (
        profile_discretization_sensitivity["profile_count"]
        / profile_discretization_sensitivity["country_count"]
    )

    profile_discretization_sensitivity["status_note"] = profile_discretization_sensitivity.apply(
        make_status_note,
        axis=1,
    )

    profile_discretization_sensitivity["is_main_set"] = profile_discretization_sensitivity["set_name"] == MAIN_SET_NAME

    profile_discretization_sensitivity = profile_discretization_sensitivity.sort_values(
        ["set_name", "shock_id", "profile_levels"]
    ).reset_index(drop=True)

main_set_discretization_sensitivity = profile_discretization_sensitivity[
    profile_discretization_sensitivity["set_name"] == MAIN_SET_NAME
].copy() if not profile_discretization_sensitivity.empty else pd.DataFrame()

profile_discretization_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_profile_discretization_all_runs.csv",
    index=False,
)

main_set_discretization_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_profile_discretization_main_set.csv",
    index=False,
)

display(main_set_discretization_sensitivity)

,run_key,set_name,shock_id,profile_levels,sample_strategy,require_recovery_available,country_count,profile_count,variable_count,variables,pareto_profile_count,pareto_country_count,dominance_relation_count,hasse_edge_count,layer_count,max_country_count_per_profile,mean_country_count_per_profile,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_run,hasse_diagram_path,pareto_share,profile_per_country_ratio,status_note,is_main_set
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",1,1,110,54,7,1,1.0000,110,190,0.3667,0.6333,False,D:\Milano Bicocca\Course Materials\1st Year\02...,0.0400,1.0000,no immediate warning,True
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,83,51,5,1,1.0000,83,217,0.2767,0.7233,False,D:\Milano Bicocca\Course Materials\1st Year\02...,0.3200,1.0000,no immediate warning,True
2,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,84,54,5,1,1.0000,84,216,0.2800,0.7200,True,D:\Milano Bicocca\Course Materials\1st Year\02...,0.3200,1.0000,no immediate warning,True
3,baseline_6_variables__shock_2019__levels_3,baseline_6_variables,2019,3,complete_case,True,32,29,6,"debt_capacity_score_0_100,employment_strength_...",8,10,125,61,5,2,1.1034,125,281,0.3079,0.6921,False,D:\Milano Bicocca\Course Materials\1st Year\02...,0.3125,0.9062,no immediate warning,True
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,2019,4,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,13,126,62,5,2,1.0323,126,339,0.2710,0.7290,False,D:\Milano Bicocca\Course Materials\1st Year\02...,0.4062,0.9688,no immediate warning,True
5,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,14,90,53,5,2,1.0323,90,375,0.1935,0.8065,True,D:\Milano Bicocca\Course Materials\1st Year\02...,0.4375,0.9688,no immediate warning,True


In [7]:
# ------------------------------------------------------
# 3. EPSILON TOLERANCE SENSITIVITY
# ------------------------------------------------------

epsilon_tolerance_sensitivity = epsilon_run_summary.copy()

if not epsilon_tolerance_sensitivity.empty:
    epsilon_tolerance_sensitivity["pareto_share"] = (
        epsilon_tolerance_sensitivity["pareto_country_count"]
        / epsilon_tolerance_sensitivity["country_count"]
    )

    epsilon_tolerance_sensitivity["validity_flag"] = np.where(
        epsilon_tolerance_sensitivity["is_valid_partial_order"].astype(str).str.lower().isin(["true", "1"]),
        "valid_partial_order",
        "invalid_review",
    )

    epsilon_tolerance_sensitivity["status_note"] = epsilon_tolerance_sensitivity.apply(
        make_status_note,
        axis=1,
    )

    epsilon_tolerance_sensitivity["is_main_set"] = epsilon_tolerance_sensitivity["set_name"] == MAIN_SET_NAME

    epsilon_tolerance_sensitivity = epsilon_tolerance_sensitivity.sort_values(
        ["set_name", "shock_id", "epsilon"]
    ).reset_index(drop=True)

main_set_epsilon_tolerance_sensitivity = epsilon_tolerance_sensitivity[
    epsilon_tolerance_sensitivity["set_name"] == MAIN_SET_NAME
].copy() if not epsilon_tolerance_sensitivity.empty else pd.DataFrame()

epsilon_tolerance_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_epsilon_tolerance_all_runs.csv",
    index=False,
)

main_set_epsilon_tolerance_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_epsilon_tolerance_main_set.csv",
    index=False,
)

display(main_set_epsilon_tolerance_sensitivity)

,run_key,set_name,shock_id,epsilon,dominance_rule,sample_strategy,require_recovery_available,country_count,variable_count,variables,dominance_relation_count,hasse_edge_count,pareto_country_count,pareto_country_set,is_dag,reciprocal_pair_count,is_valid_partial_order,layer_count,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_set,pareto_share,validity_flag,status_note,is_main_set
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,strict_continuous,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
1,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",52,45,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,52,248,0.1733,0.8267,True,0.5200,valid_partial_order,large frontier,True
2,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",64,54,10,CAN;DNK;EST;FIN;GBR;IRL;LUX;SVN;SWE;USA,True,0,True,3,64,236,0.2133,0.7867,True,0.4000,valid_partial_order,no immediate warning,True
3,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",90,57,9,CAN;DNK;EST;FIN;GBR;IRL;LUX;SWE;USA,True,0,True,5,90,210,0.3000,0.7000,True,0.3600,valid_partial_order,no immediate warning,True
4,baseline_6_variables__shock_2007__eps_0.15,baseline_6_variables,2007,0.1500,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",113,63,7,CAN;DNK;EST;FIN;LUX;SWE;USA,True,0,True,5,113,187,0.3767,0.6233,True,0.2800,valid_partial_order,no immediate warning,True
5,baseline_6_variables__shock_2007__eps_0.20,baseline_6_variables,2007,0.2000,epsilon_tolerance,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",142,50,6,CAN;DNK;EST;FIN;SWE;USA,True,0,True,7,142,158,0.4733,0.5267,True,0.2400,valid_partial_order,no immediate warning,True
6,baseline_6_variables__shock_2019__eps_0.00,baseline_6_variables,2019,0.0000,strict_continuous,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True,0.6562,valid_partial_order,large frontier,True
7,baseline_6_variables__shock_2019__eps_0.02,baseline_6_variables,2019,0.0200,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",78,58,16,AUS;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;IRL;KOR;LU...,True,0,True,4,78,418,0.1573,0.8427,True,0.5000,valid_partial_order,large frontier,True
8,baseline_6_variables__shock_2019__eps_0.05,baseline_6_variables,2019,0.0500,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",111,71,12,AUS;CAN;CHE;CZE;DNK;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,4,111,385,0.2238,0.7762,True,0.3750,valid_partial_order,no immediate warning,True
9,baseline_6_variables__shock_2019__eps_0.10,baseline_6_variables,2019,0.1000,epsilon_tolerance,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",178,59,10,AUS;CAN;CHE;EST;GBR;KOR;LUX;NZL;SWE;USA,True,0,True,6,178,318,0.3589,0.6411,True,0.3125,valid_partial_order,no immediate warning,True


In [8]:
# ------------------------------------------------------
# 4. EPSILON-MARGIN ROBUSTNESS SENSITIVITY
# ------------------------------------------------------

epsilon_margin_sensitivity = epsilon_margin_run_summary.copy()

if not epsilon_margin_sensitivity.empty:
    epsilon_margin_sensitivity["pareto_share"] = (
        epsilon_margin_sensitivity["pareto_country_count"]
        / epsilon_margin_sensitivity["country_count"]
    )

    epsilon_margin_sensitivity["validity_flag"] = np.where(
        epsilon_margin_sensitivity["is_valid_partial_order"].astype(str).str.lower().isin(["true", "1"]),
        "valid_partial_order",
        "invalid_review",
    )

    epsilon_margin_sensitivity["status_note"] = epsilon_margin_sensitivity.apply(
        make_status_note,
        axis=1,
    )

    epsilon_margin_sensitivity["is_main_set"] = epsilon_margin_sensitivity["set_name"] == MAIN_SET_NAME

    epsilon_margin_sensitivity = epsilon_margin_sensitivity.sort_values(
        ["set_name", "shock_id", "epsilon_margin"]
    ).reset_index(drop=True)

main_set_epsilon_margin_sensitivity = epsilon_margin_sensitivity[
    epsilon_margin_sensitivity["set_name"] == MAIN_SET_NAME
].copy() if not epsilon_margin_sensitivity.empty else pd.DataFrame()

epsilon_margin_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_epsilon_margin_all_runs.csv",
    index=False,
)

main_set_epsilon_margin_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_epsilon_margin_main_set.csv",
    index=False,
)

display(main_set_epsilon_margin_sensitivity)

,run_key,set_name,shock_id,epsilon_margin,dominance_rule,sample_strategy,require_recovery_available,country_count,variable_count,variables,dominance_relation_count,hasse_edge_count,pareto_country_count,pareto_country_set,is_dag,reciprocal_pair_count,is_valid_partial_order,layer_count,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_set,pareto_share,validity_flag,status_note,is_main_set
0,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,epsilon_margin_minimum_advantage,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
1,baseline_6_variables__shock_2007__margin_0.02,baseline_6_variables,2007,0.0200,epsilon_margin_minimum_advantage,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
2,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,epsilon_margin_minimum_advantage,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
3,baseline_6_variables__shock_2007__margin_0.10,baseline_6_variables,2007,0.1000,epsilon_margin_minimum_advantage,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
4,baseline_6_variables__shock_2007__margin_0.15,baseline_6_variables,2007,0.1500,epsilon_margin_minimum_advantage,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
5,baseline_6_variables__shock_2007__margin_0.20,baseline_6_variables,2007,0.2000,epsilon_margin_minimum_advantage,complete_case,True,25,6,"debt_capacity_score_0_100,employment_strength_...",46,40,13,BEL;CAN;CZE;DNK;EST;FIN;GBR;IRL;LTU;LUX;SVN;SW...,True,0,True,3,46,254,0.1533,0.8467,True,0.5200,valid_partial_order,large frontier,True
6,baseline_6_variables__shock_2019__margin_0.00,baseline_6_variables,2019,0.0000,epsilon_margin_minimum_advantage,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True,0.6562,valid_partial_order,large frontier,True
7,baseline_6_variables__shock_2019__margin_0.02,baseline_6_variables,2019,0.0200,epsilon_margin_minimum_advantage,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True,0.6562,valid_partial_order,large frontier,True
8,baseline_6_variables__shock_2019__margin_0.05,baseline_6_variables,2019,0.0500,epsilon_margin_minimum_advantage,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True,0.6562,valid_partial_order,large frontier,True
9,baseline_6_variables__shock_2019__margin_0.10,baseline_6_variables,2019,0.1000,epsilon_margin_minimum_advantage,complete_case,True,32,6,"debt_capacity_score_0_100,employment_strength_...",60,50,21,AUS;AUT;BEL;CAN;CHE;CZE;DEU;DNK;EST;FIN;GBR;HU...,True,0,True,3,60,436,0.1210,0.8790,True,0.6562,valid_partial_order,large frontier,True


In [9]:
# ------------------------------------------------------
# 5. EPSILON TOLERANCE VS EPSILON-MARGIN COMPARISON
# ------------------------------------------------------

epsilon_rule_comparison_summary = epsilon_tolerance_vs_margin_comparison.copy()

if not epsilon_rule_comparison_summary.empty and "message" not in epsilon_rule_comparison_summary.columns:
    epsilon_rule_comparison_summary = epsilon_rule_comparison_summary.sort_values(
        ["set_name", "shock_id", "epsilon_value"]
    ).reset_index(drop=True)

    epsilon_rule_comparison_summary["margin_more_conservative"] = (
        epsilon_rule_comparison_summary["margin_dominance_relation_count"]
        <= epsilon_rule_comparison_summary["tolerance_dominance_relation_count"]
    )

    epsilon_rule_comparison_summary["interpretation_note"] = np.where(
        epsilon_rule_comparison_summary["margin_more_conservative"],
        "Margin rule is more conservative or equal.",
        "Review: margin rule produced more dominance relations than tolerance.",
    )

main_set_epsilon_rule_comparison = epsilon_rule_comparison_summary[
    epsilon_rule_comparison_summary["set_name"] == MAIN_SET_NAME
].copy() if not epsilon_rule_comparison_summary.empty and "set_name" in epsilon_rule_comparison_summary.columns else pd.DataFrame()

epsilon_rule_comparison_summary.to_csv(
    PROCESSED_DIR / "sensitivity_epsilon_rule_comparison.csv",
    index=False,
)

main_set_epsilon_rule_comparison.to_csv(
    PROCESSED_DIR / "sensitivity_epsilon_rule_comparison_main_set.csv",
    index=False,
)

display(main_set_epsilon_rule_comparison)

,set_name,shock_id,epsilon_value,country_count,tolerance_dominance_relation_count,tolerance_pareto_country_count,tolerance_comparability_ratio,tolerance_is_valid_partial_order,margin_dominance_relation_count,margin_pareto_country_count,margin_comparability_ratio,margin_is_valid_partial_order,dominance_relation_difference_margin_minus_tolerance,pareto_count_difference_margin_minus_tolerance,comparability_difference_margin_minus_tolerance,margin_more_conservative,interpretation_note
0,baseline_6_variables,2007,0.0000,25,46,13,0.1533,True,46,13,0.1533,True,0,0,0.0000,True,Margin rule is more conservative or equal.
1,baseline_6_variables,2007,0.0200,25,52,13,0.1733,True,46,13,0.1533,True,-6,0,-0.0200,True,Margin rule is more conservative or equal.
2,baseline_6_variables,2007,0.0500,25,64,10,0.2133,True,46,13,0.1533,True,-18,3,-0.0600,True,Margin rule is more conservative or equal.
3,baseline_6_variables,2007,0.1000,25,90,9,0.3000,True,46,13,0.1533,True,-44,4,-0.1467,True,Margin rule is more conservative or equal.
4,baseline_6_variables,2007,0.1500,25,113,7,0.3767,True,46,13,0.1533,True,-67,6,-0.2233,True,Margin rule is more conservative or equal.
5,baseline_6_variables,2007,0.2000,25,142,6,0.4733,True,46,13,0.1533,True,-96,7,-0.3200,True,Margin rule is more conservative or equal.
6,baseline_6_variables,2019,0.0000,32,60,21,0.1210,True,60,21,0.1210,True,0,0,0.0000,True,Margin rule is more conservative or equal.
7,baseline_6_variables,2019,0.0200,32,78,16,0.1573,True,60,21,0.1210,True,-18,5,-0.0363,True,Margin rule is more conservative or equal.
8,baseline_6_variables,2019,0.0500,32,111,12,0.2238,True,60,21,0.1210,True,-51,9,-0.1028,True,Margin rule is more conservative or equal.
9,baseline_6_variables,2019,0.1000,32,178,10,0.3589,True,60,21,0.1210,True,-118,11,-0.2379,True,Margin rule is more conservative or equal.


In [10]:
# ------------------------------------------------------
# 6. RECOVERY VALIDATION SENSITIVITY
# ------------------------------------------------------

recovery_validation_sensitivity = recovery_validation_takeaway_table.copy()

if not recovery_validation_sensitivity.empty:
    recovery_validation_sensitivity["faster_frontier_signal"] = np.where(
        recovery_validation_sensitivity["frontier_minus_non_frontier_mean_recovery"] < 0,
        "frontier_faster_on_mean",
        np.where(
            recovery_validation_sensitivity["frontier_minus_non_frontier_mean_recovery"] > 0,
            "frontier_slower_on_mean",
            "no_mean_difference",
        ),
    )

    recovery_validation_sensitivity["layer_signal"] = np.where(
        recovery_validation_sensitivity["layer_vs_recovery_spearman"] > 0,
        "worse_layers_slower_recovery",
        np.where(
            recovery_validation_sensitivity["layer_vs_recovery_spearman"] < 0,
            "worse_layers_faster_recovery_review",
            "no_layer_signal",
        ),
    )

    recovery_validation_sensitivity["validation_strength_flag"] = np.where(
        (
            (recovery_validation_sensitivity["frontier_minus_non_frontier_mean_recovery"] < 0)
            & (recovery_validation_sensitivity["layer_vs_recovery_spearman"] > 0)
        ),
        "supportive",
        np.where(
            recovery_validation_sensitivity["frontier_minus_non_frontier_mean_recovery"] < 0,
            "partially_supportive",
            "weak_or_mixed",
        ),
    )

working_main_recovery_validation_sensitivity = recovery_validation_sensitivity[
    recovery_validation_sensitivity["set_name"] == MAIN_SET_NAME
].copy() if not recovery_validation_sensitivity.empty and "set_name" in recovery_validation_sensitivity.columns else pd.DataFrame()

recovery_validation_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_recovery_validation_takeaways.csv",
    index=False,
)

working_main_recovery_validation_sensitivity.to_csv(
    PROCESSED_DIR / "sensitivity_recovery_validation_main_set.csv",
    index=False,
)

display(working_main_recovery_validation_sensitivity)

,method,set_name,shock_id,setting,validation_variant,country_count,frontier_or_pareto_count,frontier_minus_non_frontier_mean_recovery,frontier_minus_non_frontier_median_recovery,layer_vs_recovery_spearman,interpretation,faster_frontier_signal,layer_signal,validation_strength_flag
0,profile_poset,baseline_6_variables,2007,5_levels,all_recovery_available,25,8,-0.8235,1.0000,0.0260,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
1,profile_poset,baseline_6_variables,2007,5_levels,affected_recovered_only_exclude_0_and_20,23,8,-0.2667,1.0000,-0.0386,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_faster_recovery_review,partially_supportive
2,profile_poset,baseline_6_variables,2019,5_levels,all_recovery_available,32,14,-0.1508,0.0000,0.2813,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
3,profile_poset,baseline_6_variables,2019,5_levels,affected_recovered_only_exclude_0_and_20,27,12,-0.2167,0.0000,0.3644,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
4,epsilon_margin,baseline_6_variables,2007,margin_0.00,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
5,epsilon_margin,baseline_6_variables,2007,margin_0.00,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
6,epsilon_margin,baseline_6_variables,2007,margin_0.05,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
7,epsilon_margin,baseline_6_variables,2007,margin_0.05,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
8,epsilon_margin,baseline_6_variables,2007,margin_0.10,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
9,epsilon_margin,baseline_6_variables,2007,margin_0.10,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive


In [11]:
# ------------------------------------------------------
# 7. ROBUSTNESS DASHBOARD TABLE
# ------------------------------------------------------

dashboard_rows = []

# Profile discretization rows.
if not main_set_discretization_sensitivity.empty:
    for _, row in main_set_discretization_sensitivity.iterrows():
        dashboard_rows.append({
            "sensitivity_family": "profile_discretization",
            "method": "profile_poset",
            "set_name": row["set_name"],
            "shock_id": row["shock_id"],
            "setting": f"{int(row['profile_levels'])}_levels",
            "country_count": row["country_count"],
            "frontier_or_pareto_count": row["pareto_country_count"],
            "frontier_or_pareto_share": row["pareto_share"],
            "dominance_or_comparability": row["comparability_ratio"],
            "layer_count": row["layer_count"],
            "validity": "valid_profile_poset",
            "main_interpretation": row["status_note"],
        })

# Epsilon tolerance rows.
if not main_set_epsilon_tolerance_sensitivity.empty:
    for _, row in main_set_epsilon_tolerance_sensitivity.iterrows():
        dashboard_rows.append({
            "sensitivity_family": "epsilon_tolerance",
            "method": "continuous_country_level",
            "set_name": row["set_name"],
            "shock_id": row["shock_id"],
            "setting": f"epsilon_{float(row['epsilon']):.2f}",
            "country_count": row["country_count"],
            "frontier_or_pareto_count": row["pareto_country_count"],
            "frontier_or_pareto_share": row["pareto_share"],
            "dominance_or_comparability": row["comparability_ratio"],
            "layer_count": row["layer_count"],
            "validity": row["validity_flag"],
            "main_interpretation": "diagnostic tolerance rule; permits small disadvantages",
        })

# Epsilon margin rows.
if not main_set_epsilon_margin_sensitivity.empty:
    for _, row in main_set_epsilon_margin_sensitivity.iterrows():
        dashboard_rows.append({
            "sensitivity_family": "epsilon_margin",
            "method": "continuous_country_level",
            "set_name": row["set_name"],
            "shock_id": row["shock_id"],
            "setting": f"margin_{float(row['epsilon_margin']):.2f}",
            "country_count": row["country_count"],
            "frontier_or_pareto_count": row["pareto_country_count"],
            "frontier_or_pareto_share": row["pareto_share"],
            "dominance_or_comparability": row["comparability_ratio"],
            "layer_count": row["layer_count"],
            "validity": row["validity_flag"],
            "main_interpretation": "safer robustness rule; no worse dimensions allowed",
        })

robustness_dashboard_table = pd.DataFrame(dashboard_rows)

robustness_dashboard_table.to_csv(
    PROCESSED_DIR / "robustness_dashboard_table.csv",
    index=False,
)

display(robustness_dashboard_table)

,sensitivity_family,method,set_name,shock_id,setting,country_count,frontier_or_pareto_count,frontier_or_pareto_share,dominance_or_comparability,layer_count,validity,main_interpretation
0,profile_discretization,profile_poset,baseline_6_variables,2007,3_levels,25,1,0.0400,0.3667,7,valid_profile_poset,no immediate warning
1,profile_discretization,profile_poset,baseline_6_variables,2007,4_levels,25,8,0.3200,0.2767,5,valid_profile_poset,no immediate warning
2,profile_discretization,profile_poset,baseline_6_variables,2007,5_levels,25,8,0.3200,0.2800,5,valid_profile_poset,no immediate warning
3,profile_discretization,profile_poset,baseline_6_variables,2019,3_levels,32,10,0.3125,0.3079,5,valid_profile_poset,no immediate warning
4,profile_discretization,profile_poset,baseline_6_variables,2019,4_levels,32,13,0.4062,0.2710,5,valid_profile_poset,no immediate warning
5,profile_discretization,profile_poset,baseline_6_variables,2019,5_levels,32,14,0.4375,0.1935,5,valid_profile_poset,no immediate warning
6,epsilon_tolerance,continuous_country_level,baseline_6_variables,2007,epsilon_0.00,25,13,0.5200,0.1533,3,valid_partial_order,diagnostic tolerance rule; permits small disad...
7,epsilon_tolerance,continuous_country_level,baseline_6_variables,2007,epsilon_0.02,25,13,0.5200,0.1733,3,valid_partial_order,diagnostic tolerance rule; permits small disad...
8,epsilon_tolerance,continuous_country_level,baseline_6_variables,2007,epsilon_0.05,25,10,0.4000,0.2133,3,valid_partial_order,diagnostic tolerance rule; permits small disad...
9,epsilon_tolerance,continuous_country_level,baseline_6_variables,2007,epsilon_0.10,25,9,0.3600,0.3000,5,valid_partial_order,diagnostic tolerance rule; permits small disad...


In [12]:
# ------------------------------------------------------
# 8. DECISION-CANDIDATE TABLE
# ------------------------------------------------------

decision_rows = []

# Discretization decision.
if not main_set_discretization_sensitivity.empty:
    for shock_id, group in main_set_discretization_sensitivity.groupby("shock_id"):
        group = group.sort_values("profile_levels")

        best_readable = group.copy()
        best_readable["pareto_share_abs_distance_from_0_30"] = (best_readable["pareto_share"] - 0.30).abs()
        best_readable = best_readable.sort_values(
            ["pareto_share_abs_distance_from_0_30", "profile_per_country_ratio"]
        )

        suggested_level = best_readable.iloc[0]["profile_levels"] if not best_readable.empty else np.nan

        decision_rows.append({
            "decision_area": "profile_discretization_level",
            "shock_id": shock_id,
            "current_working_default": WORKING_MAIN_PROFILE_LEVEL,
            "candidate_choice": suggested_level,
            "evidence": "Compare 3, 4, and 5 levels using profile count, Pareto share, comparability, and readability.",
            "risk_if_ignored": "Too many levels can make each country unique; too few levels may overcollapse profiles.",
            "decision_status": "defer_until_final_review",
        })

# Epsilon reporting decision.
if not main_set_epsilon_rule_comparison.empty:
    decision_rows.append({
        "decision_area": "epsilon_rule_for_reporting",
        "shock_id": "both",
        "current_working_default": "report epsilon-margin as robustness; tolerance as diagnostic stress test",
        "candidate_choice": "epsilon_margin",
        "evidence": "Margin rule is more conservative and preserves no-worse-in-any-dimension logic.",
        "risk_if_ignored": "Tolerance rule may be misunderstood because it permits small disadvantages.",
        "decision_status": "likely_but_defer_until_final_review",
    })

# Variable set decision.
if not variable_selection_sensitivity.empty:
    decision_rows.append({
        "decision_area": "final_variable_set",
        "shock_id": "both",
        "current_working_default": MAIN_SET_NAME,
        "candidate_choice": MAIN_SET_NAME,
        "evidence": "Baseline six variables cover debt, employment, R&D, human capital, energy security, and governance. Debt is the main coverage bottleneck.",
        "risk_if_ignored": "Dropping debt improves sample but weakens fiscal-capacity interpretation; keeping debt reduces complete-case sample.",
        "decision_status": "defer_until_final_review",
    })

# Recovery validation wording.
if not working_main_recovery_validation_sensitivity.empty:
    decision_rows.append({
        "decision_area": "recovery_validation_claim_strength",
        "shock_id": "both",
        "current_working_default": "partial support, not causal proof",
        "candidate_choice": "moderate_associational_validation",
        "evidence": "Frontier/Pareto countries generally recover faster, especially under epsilon-margin; mismatch cases remain.",
        "risk_if_ignored": "Overclaiming would imply causality or prediction, which the design does not support.",
        "decision_status": "likely_but_defer_until_final_writeup",
    })

methodological_decision_candidates = pd.DataFrame(decision_rows)

methodological_decision_candidates.to_csv(
    PROCESSED_DIR / "methodological_decision_candidates.csv",
    index=False,
)

display(methodological_decision_candidates)

,decision_area,shock_id,current_working_default,candidate_choice,evidence,risk_if_ignored,decision_status
0,profile_discretization_level,2007,5,4,"Compare 3, 4, and 5 levels using profile count...",Too many levels can make each country unique; ...,defer_until_final_review
1,profile_discretization_level,2019,5,3,"Compare 3, 4, and 5 levels using profile count...",Too many levels can make each country unique; ...,defer_until_final_review
2,epsilon_rule_for_reporting,both,report epsilon-margin as robustness; tolerance...,epsilon_margin,Margin rule is more conservative and preserves...,Tolerance rule may be misunderstood because it...,likely_but_defer_until_final_review
3,final_variable_set,both,baseline_6_variables,baseline_6_variables,"Baseline six variables cover debt, employment,...",Dropping debt improves sample but weakens fisc...,defer_until_final_review
4,recovery_validation_claim_strength,both,"partial support, not causal proof",moderate_associational_validation,Frontier/Pareto countries generally recover fa...,Overclaiming would imply causality or predicti...,likely_but_defer_until_final_writeup


In [13]:
# ------------------------------------------------------
# 9. REPORT-READY SUMMARY TABLES
# ------------------------------------------------------

# Table A: main discretization summary.
report_table_profile_sensitivity = main_set_discretization_sensitivity[
    [
        "set_name",
        "shock_id",
        "profile_levels",
        "country_count",
        "profile_count",
        "pareto_country_count",
        "pareto_share",
        "comparability_ratio",
        "incomparability_ratio",
        "layer_count",
        "status_note",
    ]
].copy() if not main_set_discretization_sensitivity.empty else pd.DataFrame()

# Table B: epsilon margin summary.
report_table_epsilon_margin = main_set_epsilon_margin_sensitivity[
    [
        "set_name",
        "shock_id",
        "epsilon_margin",
        "country_count",
        "dominance_relation_count",
        "pareto_country_count",
        "pareto_share",
        "comparability_ratio",
        "is_valid_partial_order",
        "status_note",
    ]
].copy() if not main_set_epsilon_margin_sensitivity.empty else pd.DataFrame()

# Table C: tolerance-vs-margin compact.
report_table_epsilon_rule_comparison = main_set_epsilon_rule_comparison[
    [
        "set_name",
        "shock_id",
        "epsilon_value",
        "tolerance_dominance_relation_count",
        "margin_dominance_relation_count",
        "tolerance_pareto_country_count",
        "margin_pareto_country_count",
        "tolerance_comparability_ratio",
        "margin_comparability_ratio",
        "interpretation_note",
    ]
].copy() if not main_set_epsilon_rule_comparison.empty and "interpretation_note" in main_set_epsilon_rule_comparison.columns else pd.DataFrame()

# Table D: recovery validation compact.
report_table_recovery_validation = working_main_recovery_validation_sensitivity.copy()

report_table_profile_sensitivity.to_csv(
    PROCESSED_DIR / "report_table_profile_sensitivity.csv",
    index=False,
)

report_table_epsilon_margin.to_csv(
    PROCESSED_DIR / "report_table_epsilon_margin.csv",
    index=False,
)

report_table_epsilon_rule_comparison.to_csv(
    PROCESSED_DIR / "report_table_epsilon_rule_comparison.csv",
    index=False,
)

report_table_recovery_validation.to_csv(
    PROCESSED_DIR / "report_table_recovery_validation.csv",
    index=False,
)

display(report_table_profile_sensitivity)
display(report_table_epsilon_margin)
display(report_table_epsilon_rule_comparison.head(50))
display(report_table_recovery_validation)

,set_name,shock_id,profile_levels,country_count,profile_count,pareto_country_count,pareto_share,comparability_ratio,incomparability_ratio,layer_count,status_note
0,baseline_6_variables,2007,3,25,25,1,0.0400,0.3667,0.6333,7,no immediate warning
1,baseline_6_variables,2007,4,25,25,8,0.3200,0.2767,0.7233,5,no immediate warning
2,baseline_6_variables,2007,5,25,25,8,0.3200,0.2800,0.7200,5,no immediate warning
3,baseline_6_variables,2019,3,32,29,10,0.3125,0.3079,0.6921,5,no immediate warning
4,baseline_6_variables,2019,4,32,31,13,0.4062,0.2710,0.7290,5,no immediate warning
5,baseline_6_variables,2019,5,32,31,14,0.4375,0.1935,0.8065,5,no immediate warning


,set_name,shock_id,epsilon_margin,country_count,dominance_relation_count,pareto_country_count,pareto_share,comparability_ratio,is_valid_partial_order,status_note
0,baseline_6_variables,2007,0.0000,25,46,13,0.5200,0.1533,True,large frontier
1,baseline_6_variables,2007,0.0200,25,46,13,0.5200,0.1533,True,large frontier
2,baseline_6_variables,2007,0.0500,25,46,13,0.5200,0.1533,True,large frontier
3,baseline_6_variables,2007,0.1000,25,46,13,0.5200,0.1533,True,large frontier
4,baseline_6_variables,2007,0.1500,25,46,13,0.5200,0.1533,True,large frontier
5,baseline_6_variables,2007,0.2000,25,46,13,0.5200,0.1533,True,large frontier
6,baseline_6_variables,2019,0.0000,32,60,21,0.6562,0.1210,True,large frontier
7,baseline_6_variables,2019,0.0200,32,60,21,0.6562,0.1210,True,large frontier
8,baseline_6_variables,2019,0.0500,32,60,21,0.6562,0.1210,True,large frontier
9,baseline_6_variables,2019,0.1000,32,60,21,0.6562,0.1210,True,large frontier


,set_name,shock_id,epsilon_value,tolerance_dominance_relation_count,margin_dominance_relation_count,tolerance_pareto_country_count,margin_pareto_country_count,tolerance_comparability_ratio,margin_comparability_ratio,interpretation_note
0,baseline_6_variables,2007,0.0000,46,46,13,13,0.1533,0.1533,Margin rule is more conservative or equal.
1,baseline_6_variables,2007,0.0200,52,46,13,13,0.1733,0.1533,Margin rule is more conservative or equal.
2,baseline_6_variables,2007,0.0500,64,46,10,13,0.2133,0.1533,Margin rule is more conservative or equal.
3,baseline_6_variables,2007,0.1000,90,46,9,13,0.3000,0.1533,Margin rule is more conservative or equal.
4,baseline_6_variables,2007,0.1500,113,46,7,13,0.3767,0.1533,Margin rule is more conservative or equal.
5,baseline_6_variables,2007,0.2000,142,46,6,13,0.4733,0.1533,Margin rule is more conservative or equal.
6,baseline_6_variables,2019,0.0000,60,60,21,21,0.1210,0.1210,Margin rule is more conservative or equal.
7,baseline_6_variables,2019,0.0200,78,60,16,21,0.1573,0.1210,Margin rule is more conservative or equal.
8,baseline_6_variables,2019,0.0500,111,60,12,21,0.2238,0.1210,Margin rule is more conservative or equal.
9,baseline_6_variables,2019,0.1000,178,60,10,21,0.3589,0.1210,Margin rule is more conservative or equal.


,method,set_name,shock_id,setting,validation_variant,country_count,frontier_or_pareto_count,frontier_minus_non_frontier_mean_recovery,frontier_minus_non_frontier_median_recovery,layer_vs_recovery_spearman,interpretation,faster_frontier_signal,layer_signal,validation_strength_flag
0,profile_poset,baseline_6_variables,2007,5_levels,all_recovery_available,25,8,-0.8235,1.0000,0.0260,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
1,profile_poset,baseline_6_variables,2007,5_levels,affected_recovered_only_exclude_0_and_20,23,8,-0.2667,1.0000,-0.0386,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_faster_recovery_review,partially_supportive
2,profile_poset,baseline_6_variables,2019,5_levels,all_recovery_available,32,14,-0.1508,0.0000,0.2813,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
3,profile_poset,baseline_6_variables,2019,5_levels,affected_recovered_only_exclude_0_and_20,27,12,-0.2167,0.0000,0.3644,Negative mean/median difference means Pareto c...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
4,epsilon_margin,baseline_6_variables,2007,margin_0.00,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
5,epsilon_margin,baseline_6_variables,2007,margin_0.00,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
6,epsilon_margin,baseline_6_variables,2007,margin_0.05,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
7,epsilon_margin,baseline_6_variables,2007,margin_0.05,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
8,epsilon_margin,baseline_6_variables,2007,margin_0.10,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive
9,epsilon_margin,baseline_6_variables,2007,margin_0.10,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...,frontier_faster_on_mean,worse_layers_slower_recovery,supportive


In [14]:
# ------------------------------------------------------
# 10. SENSITIVITY FIGURES
# ------------------------------------------------------

# Figure 1: Profile comparability across discretization levels.
if not main_set_discretization_sensitivity.empty:
    for shock_id, group in main_set_discretization_sensitivity.groupby("shock_id"):
        group = group.sort_values("profile_levels")

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["profile_levels"], group["comparability_ratio"], marker="o")
        ax.set_title(f"Profile comparability by discretization level — shock {shock_id}")
        ax.set_xlabel("Profile levels")
        ax.set_ylabel("Comparability ratio")
        ax.set_xticks(sorted(group["profile_levels"].unique()))
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURES_DIR / f"profile_comparability_by_levels_shock_{shock_id}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)

# Figure 2: Pareto share across discretization levels.
if not main_set_discretization_sensitivity.empty:
    for shock_id, group in main_set_discretization_sensitivity.groupby("shock_id"):
        group = group.sort_values("profile_levels")

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["profile_levels"], group["pareto_share"], marker="o")
        ax.set_title(f"Pareto share by discretization level — shock {shock_id}")
        ax.set_xlabel("Profile levels")
        ax.set_ylabel("Pareto country share")
        ax.set_xticks(sorted(group["profile_levels"].unique()))
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURES_DIR / f"profile_pareto_share_by_levels_shock_{shock_id}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)

# Figure 3: Epsilon tolerance vs margin comparability.
if not main_set_epsilon_rule_comparison.empty:
    for shock_id, group in main_set_epsilon_rule_comparison.groupby("shock_id"):
        group = group.sort_values("epsilon_value")

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["epsilon_value"], group["tolerance_comparability_ratio"], marker="o", label="Tolerance")
        ax.plot(group["epsilon_value"], group["margin_comparability_ratio"], marker="o", label="Margin")
        ax.set_title(f"Epsilon rule comparison — shock {shock_id}")
        ax.set_xlabel("Epsilon value")
        ax.set_ylabel("Comparability ratio")
        ax.legend()
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURES_DIR / f"epsilon_rule_comparison_comparability_shock_{shock_id}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)

# Figure 4: Recovery validation differences.
if not working_main_recovery_validation_sensitivity.empty:
    plot_df = working_main_recovery_validation_sensitivity.copy()
    plot_df = plot_df[
        plot_df["validation_variant"].isin([PRIMARY_RECOVERY_VARIANT, STRICT_RECOVERY_VARIANT])
    ].copy()

    if not plot_df.empty:
        for shock_id, group in plot_df.groupby("shock_id"):
            labels = group["method"].astype(str) + "\n" + group["setting"].astype(str) + "\n" + group["validation_variant"].astype(str)
            values = group["frontier_minus_non_frontier_mean_recovery"]

            fig, ax = plt.subplots(figsize=(max(8, len(group) * 1.2), 5))
            ax.bar(range(len(group)), values)
            ax.axhline(0, linewidth=1)
            ax.set_title(f"Recovery validation: frontier/Pareto minus non-frontier mean — shock {shock_id}")
            ax.set_ylabel("Mean recovery difference")
            ax.set_xticks(range(len(group)))
            ax.set_xticklabels(labels, rotation=45, ha="right")
            ax.grid(axis="y", alpha=0.25)
            fig.tight_layout()

            path = FIGURES_DIR / f"recovery_validation_mean_difference_shock_{shock_id}.png"
            fig.savefig(path, dpi=200, bbox_inches="tight")
            plt.close(fig)

print("Figures created in:")
print(FIGURES_DIR.resolve())

Figures created in:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Sensitivity_Analysis


In [15]:
# ------------------------------------------------------
# 11. FINAL SENSITIVITY SYNTHESIS
# ------------------------------------------------------

synthesis_rows = []

# Variable selection synthesis.
if not variable_selection_sensitivity.empty:
    weak_coverage_count = 0
    if "coverage_flag" in variable_selection_sensitivity.columns:
        weak_coverage_count = int((variable_selection_sensitivity["coverage_flag"] == "weak_review").sum())

    high_redundancy_count = 0
    if "redundancy_flag" in variable_selection_sensitivity.columns:
        high_redundancy_count = int((variable_selection_sensitivity["redundancy_flag"] == "high_redundancy").sum())

    synthesis_rows.append({
        "sensitivity_area": "variable_selection",
        "status": "review_required" if weak_coverage_count or high_redundancy_count else "stable_enough",
        "main_finding": f"Weak coverage variables: {weak_coverage_count}; high redundancy variables: {high_redundancy_count}. Debt capacity remains the key sample-size bottleneck if flagged in scorecard.",
        "report_implication": "Explain why the final variable set is multidimensional and why highly overlapping variables are not all included.",
    })

# Profile discretization synthesis.
if not main_set_discretization_sensitivity.empty:
    max_profile_ratio = main_set_discretization_sensitivity["profile_per_country_ratio"].max()
    max_pareto_share = main_set_discretization_sensitivity["pareto_share"].max()

    synthesis_rows.append({
        "sensitivity_area": "profile_discretization",
        "status": "sensitive_but_usable",
        "main_finding": f"Profile/country ratio ranges up to {max_profile_ratio:.2f}; Pareto share ranges up to {max_pareto_share:.2f}. Higher discretization can make profiles sparse.",
        "report_implication": "Choose profile levels based on interpretability and show robustness across 3, 4, and 5 levels.",
    })

# Epsilon tolerance synthesis.
if not main_set_epsilon_tolerance_sensitivity.empty:
    invalid_count = int((main_set_epsilon_tolerance_sensitivity["validity_flag"] != "valid_partial_order").sum())
    max_comp = main_set_epsilon_tolerance_sensitivity["comparability_ratio"].max()

    synthesis_rows.append({
        "sensitivity_area": "epsilon_tolerance",
        "status": "diagnostic_only" if invalid_count == 0 else "diagnostic_with_invalid_runs",
        "main_finding": f"Invalid runs: {invalid_count}; maximum comparability ratio among main-set runs: {max_comp:.3f}. Tolerance increases comparability because it permits small disadvantages.",
        "report_implication": "Use tolerance as a stress test, not the main robustness result.",
    })

# Epsilon margin synthesis.
if not main_set_epsilon_margin_sensitivity.empty:
    invalid_count = int((main_set_epsilon_margin_sensitivity["validity_flag"] != "valid_partial_order").sum())
    comp_min = main_set_epsilon_margin_sensitivity["comparability_ratio"].min()
    comp_max = main_set_epsilon_margin_sensitivity["comparability_ratio"].max()

    synthesis_rows.append({
        "sensitivity_area": "epsilon_margin",
        "status": "robust" if invalid_count == 0 else "review_invalid_runs",
        "main_finding": f"Invalid runs: {invalid_count}; main-set comparability stays in range {comp_min:.3f}–{comp_max:.3f}. Margin rule is conservative.",
        "report_implication": "Present epsilon-margin as the safer robustness check.",
    })

# Recovery validation synthesis.
if not working_main_recovery_validation_sensitivity.empty:
    supportive_count = int((working_main_recovery_validation_sensitivity["validation_strength_flag"] == "supportive").sum())
    total_count = len(working_main_recovery_validation_sensitivity)

    synthesis_rows.append({
        "sensitivity_area": "recovery_validation",
        "status": "partial_support",
        "main_finding": f"Supportive validation rows: {supportive_count}/{total_count}. Frontier/Pareto countries generally recover faster, but mismatch cases remain.",
        "report_implication": "Frame validation as partial associational support, not causal proof.",
    })

sensitivity_synthesis = pd.DataFrame(synthesis_rows)

sensitivity_synthesis.to_csv(
    PROCESSED_DIR / "sensitivity_synthesis.csv",
    index=False,
)

sensitivity_synthesis.to_csv(
    DIAGNOSTICS_DIR / "sensitivity_synthesis.csv",
    index=False,
)

display(sensitivity_synthesis)

,sensitivity_area,status,main_finding,report_implication
0,variable_selection,stable_enough,Weak coverage variables: 0; high redundancy va...,Explain why the final variable set is multidim...
1,profile_discretization,sensitive_but_usable,Profile/country ratio ranges up to 1.00; Paret...,Choose profile levels based on interpretabilit...
2,epsilon_tolerance,diagnostic_only,Invalid runs: 0; maximum comparability ratio a...,"Use tolerance as a stress test, not the main r..."
3,epsilon_margin,robust,Invalid runs: 0; main-set comparability stays ...,Present epsilon-margin as the safer robustness...
4,recovery_validation,partial_support,Supportive validation rows: 15/16. Frontier/Pa...,Frame validation as partial associational supp...


In [16]:
# ------------------------------------------------------
# 12. OUTPUT DATA DICTIONARY AND METHOD NOTES
# ------------------------------------------------------

sensitivity_data_dictionary = pd.DataFrame([
    {
        "table": "sensitivity_variable_selection_scorecard.csv",
        "column": "coverage_flag",
        "description": "Heuristic coverage status for each candidate variable, where available.",
    },
    {
        "table": "sensitivity_profile_discretization_main_set.csv",
        "column": "profile_per_country_ratio",
        "description": "Number of profiles divided by number of countries. High values mean sparse profiles.",
    },
    {
        "table": "sensitivity_profile_discretization_main_set.csv",
        "column": "pareto_share",
        "description": "Share of countries on the Pareto frontier for that profile run.",
    },
    {
        "table": "sensitivity_epsilon_tolerance_main_set.csv",
        "column": "validity_flag",
        "description": "Whether the tolerance-based epsilon relation remained a valid partial order.",
    },
    {
        "table": "sensitivity_epsilon_margin_main_set.csv",
        "column": "validity_flag",
        "description": "Whether the margin-based epsilon relation remained a valid partial order.",
    },
    {
        "table": "sensitivity_recovery_validation_takeaways.csv",
        "column": "validation_strength_flag",
        "description": "Heuristic label summarizing whether recovery validation supports the structural frontier interpretation.",
    },
    {
        "table": "robustness_dashboard_table.csv",
        "column": "dominance_or_comparability",
        "description": "Comparability ratio for the corresponding POSet/sensitivity setting.",
    },
    {
        "table": "methodological_decision_candidates.csv",
        "column": "decision_status",
        "description": "Whether the choice should be deferred, likely selected, or reviewed later.",
    },
    {
        "table": "sensitivity_synthesis.csv",
        "column": "report_implication",
        "description": "Short guidance for how to describe the sensitivity result in the report.",
    },
])

sensitivity_methodological_notes = pd.DataFrame([
    {
        "topic": "No automatic final decisions",
        "note": "This notebook summarizes evidence for decisions. It does not automatically choose the final profile level, final variables, or final epsilon value.",
    },
    {
        "topic": "No scalar ranking",
        "note": "Sensitivity summaries compare POSet structures but do not create a universal scalar Economic Resilience Index.",
    },
    {
        "topic": "Profile discretization",
        "note": "3, 4, and 5 levels should be judged by interpretability, frontier size, comparability, and country-profile sparsity.",
    },
    {
        "topic": "Epsilon tolerance",
        "note": "Tolerance is useful as a diagnostic stress test because it allows small disadvantages and can inflate comparability.",
    },
    {
        "topic": "Epsilon margin",
        "note": "Margin is safer for reporting because it keeps the no-worse-in-any-dimension dominance logic.",
    },
    {
        "topic": "Recovery validation",
        "note": "Recovery validation is descriptive/associational and should be framed as partial support, not causal proof.",
    },
])

sensitivity_data_dictionary.to_csv(
    PROCESSED_DIR / "sensitivity_data_dictionary.csv",
    index=False,
)

sensitivity_methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "sensitivity_methodological_notes.csv",
    index=False,
)

display(sensitivity_data_dictionary)
display(sensitivity_methodological_notes)

,table,column,description
0,sensitivity_variable_selection_scorecard.csv,coverage_flag,Heuristic coverage status for each candidate v...
1,sensitivity_profile_discretization_main_set.csv,profile_per_country_ratio,Number of profiles divided by number of countr...
2,sensitivity_profile_discretization_main_set.csv,pareto_share,Share of countries on the Pareto frontier for ...
3,sensitivity_epsilon_tolerance_main_set.csv,validity_flag,Whether the tolerance-based epsilon relation r...
4,sensitivity_epsilon_margin_main_set.csv,validity_flag,Whether the margin-based epsilon relation rema...
5,sensitivity_recovery_validation_takeaways.csv,validation_strength_flag,Heuristic label summarizing whether recovery v...
6,robustness_dashboard_table.csv,dominance_or_comparability,Comparability ratio for the corresponding POSe...
7,methodological_decision_candidates.csv,decision_status,"Whether the choice should be deferred, likely ..."
8,sensitivity_synthesis.csv,report_implication,Short guidance for how to describe the sensiti...


,topic,note
0,No automatic final decisions,This notebook summarizes evidence for decision...
1,No scalar ranking,Sensitivity summaries compare POSet structures...
2,Profile discretization,"3, 4, and 5 levels should be judged by interpr..."
3,Epsilon tolerance,Tolerance is useful as a diagnostic stress tes...
4,Epsilon margin,Margin is safer for reporting because it keeps...
5,Recovery validation,Recovery validation is descriptive/association...


In [17]:
# ------------------------------------------------------
# 13. CREATE SENSITIVITY ANALYSIS AUDIT WORKBOOK
# ------------------------------------------------------

SENSITIVITY_AUDIT_XLSX = AUDIT_DIR / "11_Sensitivity_Analysis_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_input_summary",
        "description": "Input tables loaded for sensitivity analysis.",
        "path": DIAGNOSTICS_DIR / "sensitivity_input_summary.csv",
    },
    {
        "sheet_name": "02_synthesis",
        "description": "Overall sensitivity synthesis.",
        "path": PROCESSED_DIR / "sensitivity_synthesis.csv",
    },
    {
        "sheet_name": "03_decisions",
        "description": "Candidate methodological decisions for final review.",
        "path": PROCESSED_DIR / "methodological_decision_candidates.csv",
    },
    {
        "sheet_name": "04_dashboard",
        "description": "Main robustness dashboard table.",
        "path": PROCESSED_DIR / "robustness_dashboard_table.csv",
    },
    {
        "sheet_name": "05_var_scorecard",
        "description": "Variable-selection sensitivity scorecard.",
        "path": PROCESSED_DIR / "sensitivity_variable_selection_scorecard.csv",
    },
    {
        "sheet_name": "06_var_redundancy",
        "description": "Variable redundancy pairs.",
        "path": PROCESSED_DIR / "sensitivity_variable_redundancy_pairs.csv",
    },
    {
        "sheet_name": "07_profile_sens",
        "description": "Main-set profile discretization sensitivity.",
        "path": PROCESSED_DIR / "sensitivity_profile_discretization_main_set.csv",
    },
    {
        "sheet_name": "08_eps_tolerance",
        "description": "Main-set epsilon tolerance sensitivity.",
        "path": PROCESSED_DIR / "sensitivity_epsilon_tolerance_main_set.csv",
    },
    {
        "sheet_name": "09_eps_margin",
        "description": "Main-set epsilon margin sensitivity.",
        "path": PROCESSED_DIR / "sensitivity_epsilon_margin_main_set.csv",
    },
    {
        "sheet_name": "10_eps_compare",
        "description": "Tolerance vs margin comparison.",
        "path": PROCESSED_DIR / "sensitivity_epsilon_rule_comparison_main_set.csv",
    },
    {
        "sheet_name": "11_recovery",
        "description": "Recovery validation sensitivity.",
        "path": PROCESSED_DIR / "sensitivity_recovery_validation_main_set.csv",
    },
    {
        "sheet_name": "12_report_profile",
        "description": "Report-ready profile sensitivity table.",
        "path": PROCESSED_DIR / "report_table_profile_sensitivity.csv",
    },
    {
        "sheet_name": "13_report_margin",
        "description": "Report-ready epsilon-margin table.",
        "path": PROCESSED_DIR / "report_table_epsilon_margin.csv",
    },
    {
        "sheet_name": "14_report_recovery",
        "description": "Report-ready recovery validation table.",
        "path": PROCESSED_DIR / "report_table_recovery_validation.csv",
    },
    {
        "sheet_name": "15_dictionary",
        "description": "Sensitivity analysis data dictionary.",
        "path": PROCESSED_DIR / "sensitivity_data_dictionary.csv",
    },
    {
        "sheet_name": "16_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "sensitivity_methodological_notes.csv",
    },
]

used_sheets = set()

with pd.ExcelWriter(SENSITIVITY_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "11 Sensitivity Analysis Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for consolidated sensitivity analysis.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Sensitivity audit workbook created:")
print(SENSITIVITY_AUDIT_XLSX.resolve())

Sensitivity audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\11_Sensitivity_Analysis_Audit.xlsx


In [18]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("11 SENSITIVITY ANALYSIS COMPLETE")
print("=" * 80)

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nFigures folder:")
print(FIGURES_DIR.resolve())

print("\nAudit workbook:")
print(SENSITIVITY_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "sensitivity_synthesis.csv",
    "methodological_decision_candidates.csv",
    "robustness_dashboard_table.csv",
    "sensitivity_variable_selection_scorecard.csv",
    "sensitivity_profile_discretization_main_set.csv",
    "sensitivity_epsilon_tolerance_main_set.csv",
    "sensitivity_epsilon_margin_main_set.csv",
    "sensitivity_epsilon_rule_comparison_main_set.csv",
    "sensitivity_recovery_validation_main_set.csv",
    "report_table_profile_sensitivity.csv",
    "report_table_epsilon_margin.csv",
    "report_table_epsilon_rule_comparison.csv",
    "report_table_recovery_validation.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nSensitivity synthesis:")
display(sensitivity_synthesis)

print("\nMethodological decision candidates:")
display(methodological_decision_candidates)

print("\nImportant notes:")
print("1. This notebook does not make final decisions automatically.")
print("2. It consolidates variable, discretization, epsilon, margin, and recovery sensitivity.")
print("3. Use the report-ready tables for the sensitivity/report section.")
print("4. Next step is final visual/report preparation.")

print("\nNext notebook:")
print("12_Final_Visuals.ipynb")

11 SENSITIVITY ANALYSIS COMPLETE

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Sensitivity_Analysis

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\11_Sensitivity_Analysis

Figures folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Sensitivity_Analysis

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\11_Sensitivity_Analysis_Audit.xlsx

Main processed outputs:
- FOUND: sensitivity_synthesis.csv
- FOUND: methodological_decision_candidates.csv
- FOUND: robustness_dashboard_table.csv
- FOUND: sensitivity_variable_selection_scorecard.csv
- FOUND: sensitivity_profile_discretization_main_set.csv
- FOUND: sensitivity_e

,sensitivity_area,status,main_finding,report_implication
0,variable_selection,stable_enough,Weak coverage variables: 0; high redundancy va...,Explain why the final variable set is multidim...
1,profile_discretization,sensitive_but_usable,Profile/country ratio ranges up to 1.00; Paret...,Choose profile levels based on interpretabilit...
2,epsilon_tolerance,diagnostic_only,Invalid runs: 0; maximum comparability ratio a...,"Use tolerance as a stress test, not the main r..."
3,epsilon_margin,robust,Invalid runs: 0; main-set comparability stays ...,Present epsilon-margin as the safer robustness...
4,recovery_validation,partial_support,Supportive validation rows: 15/16. Frontier/Pa...,Frame validation as partial associational supp...



Methodological decision candidates:


,decision_area,shock_id,current_working_default,candidate_choice,evidence,risk_if_ignored,decision_status
0,profile_discretization_level,2007,5,4,"Compare 3, 4, and 5 levels using profile count...",Too many levels can make each country unique; ...,defer_until_final_review
1,profile_discretization_level,2019,5,3,"Compare 3, 4, and 5 levels using profile count...",Too many levels can make each country unique; ...,defer_until_final_review
2,epsilon_rule_for_reporting,both,report epsilon-margin as robustness; tolerance...,epsilon_margin,Margin rule is more conservative and preserves...,Tolerance rule may be misunderstood because it...,likely_but_defer_until_final_review
3,final_variable_set,both,baseline_6_variables,baseline_6_variables,"Baseline six variables cover debt, employment,...",Dropping debt improves sample but weakens fisc...,defer_until_final_review
4,recovery_validation_claim_strength,both,"partial support, not causal proof",moderate_associational_validation,Frontier/Pareto countries generally recover fa...,Overclaiming would imply causality or predicti...,likely_but_defer_until_final_writeup



Important notes:
1. This notebook does not make final decisions automatically.
2. It consolidates variable, discretization, epsilon, margin, and recovery sensitivity.
3. Use the report-ready tables for the sensitivity/report section.
4. Next step is final visual/report preparation.

Next notebook:
12_Final_Visuals.ipynb
